In [1]:
import os
from datasets import load_dataset
from dotenv import load_dotenv
import chess
import math
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import numpy as np
from itertools import islice
import zstandard as zstd

load_dotenv()

/home/nkminion/miniconda3/envs/PyTorchVenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
TOKEN = os.getenv("HFREAD")

if TOKEN is None:
	raise ValueError("Token not found")

Dataset = load_dataset(
	"Lichess/chess-position-evaluations",
	split='train',
	streaming=True,
	token=TOKEN
)

print('Dataset Loaded')

Dataset Loaded


In [3]:
def ProcessChessData(FENString,CPScore,MateScore):
	tensor = np.zeros((16,8,8), dtype=np.float32)
	board = chess.Board(FENString)

	PieceToLayer = {
		'P':0,'N':1,'B':2,'R':3,'Q':4,'K':5,
		'p':6,'n':7,'b':8,'r':9,'q':10,'k':11
	}

	for square in chess.SQUARES:
		piece = board.piece_at(square)

		if piece:
			symbol = piece.symbol()
			layer = PieceToLayer[symbol]

			row = 7-(square//8)
			col = square%8

			tensor[layer,row,col] = 1.0

	if board.turn == chess.WHITE:
		tensor[12,:,:] = 1.0

	if board.has_kingside_castling_rights(chess.WHITE):
		tensor[13,7,7] = 1.0
	if board.has_queenside_castling_rights(chess.WHITE):
		tensor[13,7,0] = 1.0
	if board.has_kingside_castling_rights(chess.BLACK):
		tensor[13,0,7] = 1.0
	if board.has_queenside_castling_rights(chess.BLACK):
		tensor[13,0,0] = 1.0

	for i in range(8):
		tensor[14,i,:] = (1.0/7)*(i)
		tensor[15,:,i] = (1.0/7)*(i)

	if MateScore is not None:
		TargetScore = 1.0 if MateScore > 0 else -1.0
	else:
		CPScore = CPScore if CPScore is not None else 0
		TargetScore = math.tanh(CPScore/200.0)

	TargetScore = np.array([TargetScore],dtype=np.float32)

	return tensor,TargetScore

In [4]:
class ChessIterableDataset(IterableDataset):
	def __init__(self,hfDataset):
		self.hfDataset = hfDataset

	def __iter__(self):
		WorkerInfo = get_worker_info()
		if WorkerInfo is None:
			start = 0
			step = 1
		else:
			start = WorkerInfo.id
			step = WorkerInfo.num_workers

		ShardedStream = islice(self.hfDataset,start,None,step)

		for data in ShardedStream:
			inputs,target = ProcessChessData(data['fen'],data['cp'],data['mate'])

			yield inputs,target

In [5]:
TrainDataset = ChessIterableDataset(Dataset)
TrainLoader = DataLoader(
	TrainDataset,
	batch_size=1024,
	num_workers=4,
	pin_memory=True,
	prefetch_factor=1,
	persistent_workers=True
)
TotalBoards = 0
TotalSize = 0
cctx = zstd.ZstdCompressor(level=10)
with open ('ChessData.zst','wb') as ResultFile:
	with cctx.stream_writer(ResultFile) as stream:
		for BatchIdx,(inputs,targets) in enumerate(TrainLoader):
			TotalBoards += len(targets)
			inputs = inputs.numpy().tobytes()
			targets = targets.numpy().tobytes()
			CombinedBytes = inputs+targets
			TotalSize += len(CombinedBytes)
			stream.write(CombinedBytes)
			
			if (BatchIdx+1) % 1000 == 0:
				print(f'Processed {BatchIdx+1} Batches | Uncompressed Size (Bytes): {TotalSize} | Compressed Size (Bytes): {ResultFile.tell()}')

print(f'Completed! | Total Batches: {TotalBoards//1024} | Number of boards: {TotalBoards} | Uncompressed Size: {TotalSize}')

Processed 1000 Batches | Uncompressed Size (Bytes): 4198400000 | Compressed Size (Bytes): 40515759
Processed 2000 Batches | Uncompressed Size (Bytes): 8396800000 | Compressed Size (Bytes): 81556668
Processed 3000 Batches | Uncompressed Size (Bytes): 12595200000 | Compressed Size (Bytes): 123008544
Processed 4000 Batches | Uncompressed Size (Bytes): 16793600000 | Compressed Size (Bytes): 164647674
Processed 5000 Batches | Uncompressed Size (Bytes): 20992000000 | Compressed Size (Bytes): 206463955
Processed 6000 Batches | Uncompressed Size (Bytes): 25190400000 | Compressed Size (Bytes): 248094692
Processed 7000 Batches | Uncompressed Size (Bytes): 29388800000 | Compressed Size (Bytes): 290235964
Processed 8000 Batches | Uncompressed Size (Bytes): 33587200000 | Compressed Size (Bytes): 332777022
Processed 9000 Batches | Uncompressed Size (Bytes): 37785600000 | Compressed Size (Bytes): 374657533
Processed 10000 Batches | Uncompressed Size (Bytes): 41984000000 | Compressed Size (Bytes): 416

'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/3135c379f8d7e81c4fad71a2be6f5778039cc0a1/data/train-00002-of-00017.parquet
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/3135c379f8d7e81c4fad71a2be6f5778039cc0a1/data/train-00000-of-00017.parquet
Retrying in 1s [Retry 1/5].
Retrying in 1s [Retry 1/5].


Processed 41000 Batches | Uncompressed Size (Bytes): 172134400000 | Compressed Size (Bytes): 1754332940
Processed 42000 Batches | Uncompressed Size (Bytes): 176332800000 | Compressed Size (Bytes): 1798392622
Processed 43000 Batches | Uncompressed Size (Bytes): 180531200000 | Compressed Size (Bytes): 1842401438
Processed 44000 Batches | Uncompressed Size (Bytes): 184729600000 | Compressed Size (Bytes): 1886413414
Processed 45000 Batches | Uncompressed Size (Bytes): 188928000000 | Compressed Size (Bytes): 1930677038
Processed 46000 Batches | Uncompressed Size (Bytes): 193126400000 | Compressed Size (Bytes): 1974801357
Processed 47000 Batches | Uncompressed Size (Bytes): 197324800000 | Compressed Size (Bytes): 2019079074
Processed 48000 Batches | Uncompressed Size (Bytes): 201523200000 | Compressed Size (Bytes): 2063270064
Processed 49000 Batches | Uncompressed Size (Bytes): 205721600000 | Compressed Size (Bytes): 2108458377
Processed 50000 Batches | Uncompressed Size (Bytes): 20992000000

KeyboardInterrupt: 

Current File size: 2.2GiB

Total Dataset size: 40.61GB
Estimated compressed size of total dataset: 31.01GiB